# 04 — Evaluation: gold labels, decision accuracy, and catching regressions

**What you'll learn**

- Why the shop's gold labels are computed, not opinions: `shoplab.rules.decide` is the spec, and all 40 labels recompute from it
- Score one prediction on three axes with `score_ticket` — decision, policy evidence, amount within a cent — and the `None` semantics
- Build the harness core yourself as `evaluate_mini`, then run chapter 02's agent over the dev split with the shipped `evaluate` and read four accuracies plus a cost figure
- Manufacture a regression, then catch it: named runs, `compare_runs`, and the tickets that flipped
- Replay a failing ticket with `on_step` and name the failure mode from the transcript, not the average

*Time: ~12 min on a first live run (the baseline evaluation alone is ~5 min); ~2 min cached. Cost: ~$0.02. Cached reruns are free.*

## Without a suite, an agent is vibes

Chapter 02 ended with the agent triaging one ticket and you eyeballing the answer. Eyeballing does not scale, and worse, it does not accumulate: change the prompt tomorrow and you either re-check everything by hand or trust your memory of what used to work. Until a change moves a number, every edit to a prompt, tool, or model is a guess about whether things got better.

The ops desk is unusually lucky here. Ticket triage has a written specification — the refund cascade of docs/WORLD.md, implemented in `shoplab.rules` — so every ticket carries a gold label and grading is exact comparison of strings and numbers. No rubric, no annotator panel, no second model grading the first. Open-ended output does eventually force a model into the grader's seat: *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena* (Zheng et al., 2023, [arXiv:2306.05685](https://arxiv.org/abs/2306.05685)) established that pattern and measured its biases, and it arrives in chapter 06. This chapter is exact-match country — and exact match is what you should reach for whenever the task allows it.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

### Phoenix observability (optional)

The frozen cell finds or starts the local Phoenix server and traces every LiteLLM call this notebook makes — around a hundred once the two evaluations below run. Browsing an evaluation as a list of traces is a useful second view of the harness. Optional: skip it and nothing else changes.

In [ ]:
# optional: Phoenix tracing (see notebook 03)
import obs

obs.enable_phoenix()

## The spec was there all along

Every gold label in `data/tickets.json` was computed by one function: `shoplab.rules.decide`, a 9-step first-match-wins cascade over the ticket, its order, and its customer. It returns the same shape the agent's `finish` tool emits — `{"decision", "policy_id", "refund_usd"}` — with `refund_usd` rounded to cents and `None` whenever no money moves (replacement, deny, escalate). `item_value` is the ordered line's `qty * unit_price_usd`; a sku that is not on the order raises `ValueError` rather than guessing. Each branch cites the policy document it implements:

| Step | Fires when | Policy cited | Outcome |
|---|---|---|---|
| 1 | customer is a flagged serial returner | pol-fraud | escalate |
| 2 | damaged/defective claim over $75, no photo | pol-fraud / pol-damaged | escalate over $300, else deny |
| 3 | damaged in transit | pol-damaged / pol-returns | within 30 days: item value + full shipping; later: deny |
| 4 | defective | pol-defective | within 365 days: replacement if requested, else refund; later: deny |
| 5 | anything else past 30 days | pol-returns | deny |
| 6 | wants a replacement | pol-exchanges | replacement |
| 7 | wants store credit | pol-store-credit | credit for full item value |
| 8 | refund, unopened | pol-returns | full refund |
| 9 | refund, opened | pol-restocking | vip: full refund; else 90 percent |

The next cell is pasted from `src/shoplab/rules.py`, not retyped: the sentinel comments mark the region the validator byte-compares against the package on every build. Read it top to bottom once — the ordering is load-bearing. A serial returner with a genuinely damaged item still escalates, because step 1 runs first.

In [ ]:
RETURN_WINDOW_DAYS = 30    # pol-returns
WARRANTY_DAYS = 365        # pol-defective
RESTOCKING_FEE = 0.10      # pol-restocking (waived for vip: pol-loyalty)
PHOTO_THRESHOLD_USD = 75   # pol-damaged
ESCALATE_VALUE_USD = 300   # pol-fraud

# >>> shoplab.rules.decide
def decide(ticket: dict, order: dict, customer: dict) -> dict:
    """Apply the 9-step first-match-wins cascade from docs/WORLD.md."""
    line = next((i for i in order["items"] if i["sku"] == ticket["sku"]), None)
    if line is None:
        raise ValueError(
            f"sku {ticket['sku']!r} is not an item of order {order['order_id']!r}"
        )
    item_value = round(ticket["qty"] * line["unit_price_usd"], 2)

    condition = ticket["item_condition"]
    action = ticket["requested_action"]
    days = ticket["days_since_delivery"]

    def result(decision, policy_id, refund_usd):
        return {"decision": decision, "policy_id": policy_id, "refund_usd": refund_usd}

    # 1. Flagged serial returners always go to a human. (pol-fraud)
    if customer["serial_returner"]:
        return result("escalate", "pol-fraud", None)

    # 2. Damage/defect claims over $75 need a photo; big unevidenced claims
    #    escalate, the rest are denied. (pol-fraud / pol-damaged)
    if (condition in ("damaged", "defective")
            and item_value > PHOTO_THRESHOLD_USD
            and not ticket["evidence_photo"]):
        if item_value > ESCALATE_VALUE_USD:
            return result("escalate", "pol-fraud", None)
        return result("deny", "pol-damaged", None)

    # 3. Damaged in transit: item price plus full original shipping, inside
    #    the return window. (pol-damaged / pol-returns)
    if condition == "damaged":
        if days <= RETURN_WINDOW_DAYS:
            return result("approve_refund", "pol-damaged",
                          round(item_value + order["shipping_usd"], 2))
        return result("deny", "pol-returns", None)

    # 4. Defective: 365-day warranty, replacement or refund. (pol-defective)
    if condition == "defective":
        if days <= WARRANTY_DAYS:
            if action == "replacement":
                return result("replacement", "pol-defective", None)
            return result("approve_refund", "pol-defective", item_value)
        return result("deny", "pol-defective", None)

    # 5. Change-of-mind requests past the return window. (pol-returns)
    if days > RETURN_WINDOW_DAYS:
        return result("deny", "pol-returns", None)

    # 6. Replacement for a non-defective item. (pol-exchanges)
    if action == "replacement":
        return result("replacement", "pol-exchanges", None)

    # 7. Store credit: full item value, restocking fee waived. (pol-store-credit)
    if action == "store_credit":
        return result("store_credit", "pol-store-credit", item_value)

    # Only requested_action == "refund" reaches this point.
    # 8. Unopened refund: full item value. (pol-returns)
    if condition == "unopened":
        return result("approve_refund", "pol-returns", item_value)

    # 9. Opened refund: 10% restocking fee, waived for vip. (pol-restocking)
    if customer["tier"] == "vip":
        return result("approve_refund", "pol-restocking", item_value)
    return result("partial_refund", "pol-restocking",
                  round(item_value * (1 - RESTOCKING_FEE), 2))
# <<< shoplab.rules.decide

In [ ]:
from shoplab.world import load_customers, load_orders, load_tickets

orders = {o["order_id"]: o for o in load_orders()}
customers = {c["customer_id"]: c for c in load_customers()}
tickets = load_tickets()

t = next(t for t in tickets["train"] if t["ticket_id"] == "TKT-2205")
print("recomputed: ", decide(t, orders[t["order_id"]], customers[t["customer_id"]]))
print("stored gold:", t["gold"])

mismatches = sum(decide(t, orders[t["order_id"]], customers[t["customer_id"]]) != t["gold"]
                 for split in tickets.values() for t in split)
print("tickets checked:", sum(map(len, tickets.values())), " mismatches:", mismatches)

> **What you should see:** the recomputed TKT-2205 dict equals its stored gold — `partial_refund` under `pol-restocking` for 170.99, which is 0.90 * 189.99 rounded to cents — and across all 40 tickets, zero mismatches. That zero is the whole argument: these labels are not annotator opinions that age, they are cached outputs of the spec. `scripts/check_data.py` reruns this recomputation on every build and hard-fails on any mismatch, so the data cannot drift from the rules without the build going red.

## Scoring one prediction: three axes and a cent of slack

A triage decision has three parts, and they fail independently, so score them independently: the decision (what the customer gets), the policy id (the evidence cited for it), and the amount (the money that moves). Exact string equality is right for the first two. For the amount it is a trap — the agent is allowed to do its own arithmetic, and floats do what floats do.

In [ ]:
gold = {"decision": "partial_refund", "policy_id": "pol-restocking", "refund_usd": 170.99}
pred = {"decision": "partial_refund", "policy_id": "pol-restocking",
        "refund_usd": 189.99 * 0.90}            # the model did its own arithmetic
print("pred amount:  ", pred["refund_usd"])
print("naive ==      ", pred["refund_usd"] == gold["refund_usd"])
print("within a cent:", abs(pred["refund_usd"] - gold["refund_usd"]) <= 0.01)

for a, b in [(None, None), (None, 260.10), (0.0, None)]:
    amount = (a is None and b is None) if (a is None or b is None) else abs(a - b) <= 0.01
    print(f"pred={a!r:>6}  gold={b!r:>6}  ->  amount_correct={amount}")

> **What you should see:** the model's own multiplication lands a hair off 170.99, so naive equality flunks a correct refund; a one-cent tolerance absorbs float noise without forgiving real mistakes (a missed restocking fee is off by dollars, not by 1e-13). And `None` is not zero: both sides absent is a match, but 0.0 against a gold of `None` fails — "no refund is due" and "a refund of zero dollars" are different claims.

That is the entire scorer, plus one guard you have not needed yet: a prediction that is not a dict at all — the agent answered in prose, or hit `max_steps` and returned `None` — scores zero on every axis instead of crashing the harness. You will see that guard earn its keep later in this chapter. The final form is `shoplab.evals.score_ticket`, pasted below between sentinels, byte-identical to the package. `exact` is the conjunction of the three axes — the only number that says a ticket is fully right.

In [ ]:
# >>> shoplab.evals.score_ticket
def score_ticket(pred, gold) -> dict:
    """Field-by-field scores for one predicted decision against its gold label."""
    if not isinstance(pred, dict):
        return {"decision_correct": False, "policy_correct": False,
                "amount_correct": False, "exact": False}
    decision = pred.get("decision") == gold["decision"]
    policy = pred.get("policy_id") == gold["policy_id"]
    a, b = pred.get("refund_usd"), gold["refund_usd"]
    if a is None or b is None:
        amount = a is None and b is None        # both absent is a match
    elif not isinstance(a, (int, float)) or isinstance(a, bool):
        amount = False                          # "170.99" is a schema violation, not a match
    else:
        amount = abs(a - b) <= 0.01             # a cent of float slack
    return {"decision_correct": decision, "policy_correct": policy,
            "amount_correct": amount, "exact": decision and policy and amount}
# <<< shoplab.evals.score_ticket

In [ ]:
gold = {"decision": "partial_refund", "policy_id": "pol-restocking", "refund_usd": 170.99}
preds = [
    {"decision": "partial_refund", "policy_id": "pol-restocking", "refund_usd": 170.991},
    {"decision": "partial_refund", "policy_id": "pol-loyalty", "refund_usd": 170.99},
    {"decision": "approve_refund", "policy_id": "pol-restocking", "refund_usd": 189.99},
    {"decision": "partial_refund", "policy_id": "pol-restocking"},    # amount omitted
    None,                                                             # hit max_steps
]
for pred in preds:
    print(score_ticket(pred, gold), "<-", str(pred)[:60])

> **What you should see:** only the first row is `exact` — float noise inside a cent still scores. Then, one axis at a time: a wrong policy with the right decision fails only `policy_correct` (right outcome, wrong evidence — a failure a human skimming refund amounts would never catch); a full refund where a partial was due fails decision and amount together; an omitted amount against a numeric gold fails `amount_correct`, because `.get` turns absence into `None`; and a non-dict scores zero across the board.

## Twelve tickets, four numbers

The candidate is chapter 02's agent, unchanged: `run_agent`, `standard_tools()`, and chapter 02's exact system prompt and ticket renderer — retyped below on purpose, so that what this chapter grades is stated on this page instead of imported as hidden context. A fresh toolset per ticket keeps anything from leaking between runs.

The harness, meanwhile, is machinery you already own. Its whole idea is a loop: call the candidate on every ticket, grade each prediction with `score_ticket`, average the grades. The cell after next writes exactly that — `evaluate_mini`, six working lines — and points it at three dev tickets, because the namesake machinery of an evaluation chapter is the one thing you should not accept as a bare import.

The shipped form, `shoplab.evals.evaluate(run_fn, tickets, name=...)`, keeps that loop and adds the bookkeeping: four accuracies — `decision_acc`, `policy_acc`, `amount_acc`, `exact_acc` — plus the run's LLM spend, measured as the new rows in `shoplab.llm.LEDGER`. It prints one summary line and writes everything, per-ticket rows included, to `artifacts/runs/<name>.json`. The name is not decoration; it is the key this chapter's ending turns on. The dev split (12 tickets) is the working set — small enough to rerun on every idea. Train (20) is for fitting prompts when you need more signal; test (8) stays untouched until the end of the course. The first run is live and takes a few minutes; the disk cache makes every rerun fast, free, and reproducible.

In [ ]:
SYSTEM = ("You are the operations desk agent for Larkspur Outfitters. "
          "Use the tools to look up the order, the customer, and the relevant "
          "policy before deciding. You have a hard budget of six tool calls, so "
          "look nothing up twice, and keep any commentary to one short sentence "
          "per step. When you are sure, call finish with decision "
          "(approve_refund|partial_refund|replacement|store_credit|deny|escalate), "
          "policy_id, and refund_usd (number or null). "
          "Decisions must follow shop policy, not sympathy.")

def render_ticket(t):
    return (f"Ticket {t['ticket_id']} from {t['customer_id']} about order "
            f"{t['order_id']}, sku {t['sku']}, qty {t['qty']}, "
            f"condition {t['item_condition']}, days since delivery "
            f"{t['days_since_delivery']}, photo evidence {t['evidence_photo']}, "
            f"requested action {t['requested_action']}. "
            f"Customer writes: {t['reason_text']}")

In [ ]:
from shoplab.loop import run_agent
from shoplab.tools import standard_tools

def run_ticket(ticket):
    result = run_agent(render_ticket(ticket), standard_tools(),
                       system=SYSTEM, max_steps=10)
    return result.answer

def evaluate_mini(run_fn, tickets):
    scores = [score_ticket(run_fn(t), t["gold"]) for t in tickets]
    return round(sum(s["exact"] for s in scores) / len(scores), 4)

print("exact accuracy over 3 dev tickets:", evaluate_mini(run_ticket, tickets["dev"][:3]))

> **What you should see:** one number, and on our frozen run it is 1.0 — three tickets in, every prediction exactly right. The pipeline is entirely yours: chapter 02's loop produced the predictions, this chapter's `score_ticket` graded them, a list comprehension averaged the grades. That is the eval harness, built. What the shipped `evaluate` adds is bookkeeping — all four accuracy axes, cost, per-ticket rows, and a saved artifact under a run *name* — and the bookkeeping is what the rest of the chapter runs on.

In [ ]:
import time
from shoplab.evals import evaluate

dev = tickets["dev"]
t0 = time.time()
baseline = evaluate(run_ticket, dev, name="baseline")
print(f"wall time: {time.time() - t0:.0f} s")

> **What you should see:** four accuracies well above the 1/6 floor of guessing among six decisions — for a deepseek-class model at temperature 0 expect decision accuracy roughly in the 0.6-1.0 band, and our frozen run posts a perfect 1.00 card. Do not over-read a strong card: with n=12 one ticket is worth 8 points, and temperature 0 narrows variance without eliminating it, so the same agent on a cold cache can drop a ticket or two. When a card does slip, expect the amount axis to go first, concentrated where the arithmetic is multi-step — restocking fees and shipping add-ons leave far more room for slips than flat full refunds. The run costs a cent or two; wall time is minutes live and seconds once the disk cache holds every call.

## Break it on purpose: the lazy prompt

An eval you run once is a demo. The value shows up on the second run, after a change — which is why runs carry names. So make a change of the kind that actually ships. The agent looks slow and chatty: every ticket burns six or seven tool calls to answer a three-line request, and most of what it fetches — prices, tiers, policy text — feels like things a competent model already knows. The obvious cost optimization: stop looking things up.

The variant below deletes the baseline's *look everything up first* clause and writes the opposite: no lookups, decide from the ticket text alone, lean on your knowledge of standard retail policy. Same model, same tools on offer, same tickets, same temperature — one sentence of prompt changed. If the suite is worth anything, it should notice.

In [ ]:
LAZY = ("You are the operations desk agent for Larkspur Outfitters. "
        "Tool calls are slow and expensive. Do not look anything up: no "
        "get_order, no get_customer, no search_policy, no calc. Decide from "
        "the ticket text alone, using your knowledge of standard retail "
        "policy, then call finish immediately with decision "
        "(approve_refund|partial_refund|replacement|store_credit|deny|escalate), "
        "policy_id, and refund_usd (number or null). "
        "Decisions must follow shop policy, not sympathy.")

def run_ticket_lazy(ticket):
    result = run_agent(render_ticket(ticket), standard_tools(),
                       system=LAZY, max_steps=10)
    return result.answer

lazy = evaluate(run_ticket_lazy, dev, name="lazy-prompt")

In [ ]:
from shoplab.evals import compare_runs

deltas = compare_runs("baseline", "lazy-prompt")

> **What you should see:** the change is a catastrophe with one flattering row. Decision accuracy drops by roughly a third; policy accuracy collapses toward the floor; every amount that was an actual number is now wrong or missing (the surviving `amount_acc` is all `None`-against-`None` tickets); exact accuracy lands near 0.2. Meanwhile `cost_usd` falls by about two thirds — the metric the "optimization" targeted genuinely improved, which is exactly why cost dashboards alone do not catch this class of regression. Below the table, `compare_runs` names the flipped tickets — around ten of twelve, each one a case you can go read. The tickets that survive are the deny-past-the-window ones, the only decisions readable straight off the ticket text.

This is the chapter's core habit: every prompt edit, tool change, or model bump reruns the suite under a new name before it ships. That is continuous integration for agents — the suite is cheap, the runs are named, and "it felt fine on the ticket I tried" stops being an argument.

## Read the transcript, not the score

Averages say that it broke; per-ticket rows say where; only the transcript says why. First the rows, as a dataframe.

In [ ]:
import pandas as pd

rows = [{"ticket_id": r["ticket_id"], "gold": r["gold"]["decision"],
         "pred": r["pred"].get("decision") if isinstance(r["pred"], dict) else None,
         **r["scores"]}
        for r in lazy["rows"]]
df = pd.DataFrame(rows)
print(df.to_string(index=False))
print("\nfailures by gold decision:")
print(df[~df["exact"]].groupby("gold").size().to_string())

> **What you should see:** failures spread over every gold decision except `deny`. Two rows have no predicted decision at all (pandas renders the gap as `NaN`) — the model ignored the `finish` contract and answered in markdown prose, and `score_ticket`'s non-dict guard zeroed those tickets instead of crashing the run. Elsewhere the pattern is decision-right-but-wrong-evidence: `decision_correct` True while `policy_correct` fails.

Now one failing ticket end to end. Rerun it with an `on_step` hook printing each turn. The disk cache matters here: the replay hits the same cached responses the evaluation saw, so this transcript is not a fresh roll of the dice — it is the graded run, reprinted for free.

In [ ]:
flipped = [r["ticket_id"] for r in lazy["rows"] if not r["scores"]["exact"]]
bad = next(t for t in dev if t["ticket_id"] == flipped[0])
print(render_ticket(bad), "\n")

def show(step, msg):
    calls = [f"{tc.function.name}({tc.function.arguments})"
             for tc in msg.tool_calls or []]
    print(f"step {step}:", "; ".join(calls) or (msg.content or "")[:80])

replay = run_agent(render_ticket(bad), standard_tools(), system=LAZY,
                   max_steps=10, on_step=show)
print("\npred:", replay.answer)
print("gold:", bad["gold"])

> **What you should see:** one step, zero lookups — `finish` called immediately, with an invented policy id (our frozen run cites `pol-returns-abuse`, which exists nowhere in the shop's twelve documents). On this ticket the decision is still `escalate`, and still right, but for the wrong reason: the customer happened to confess in prose to being a frequent returner, while the `serial_returner` flag that actually decides step 1 of the cascade sits unread in the customer record. Name the failure mode precisely: fabricated evidence in place of retrieved evidence. The policy axis caught it; a decision-only metric would have scored this ticket perfect.

## Runs are artifacts, not variables

Everything `evaluate` computed is already on disk, keyed by run name. That was the point of `name=`: a comparison you can only make while the right variables happen to be alive in a kernel is not evidence, it is folklore.

In [ ]:
from shoplab.evals import RUNS_DIR, load_run

for path in sorted(RUNS_DIR.glob("*.json")):
    print(f"{path.name:<22} {path.stat().st_size:>6,} bytes")

run = load_run("baseline")
print("\nkeys:", sorted(run.keys()))
r = run["rows"][0]
print("one row:", r["ticket_id"], r["pred"], r["scores"]["exact"])

> **What you should see:** one JSON file per named run, a few kilobytes each, holding the aggregate metrics plus every per-ticket prediction, gold, and score. `compare_runs` never touched the `baseline` or `lazy` variables above — it reloaded both files from disk, which is exactly what a CI job, a teammate, or you-next-week would do.

That is why runs are artifacts: a variable dies with the kernel and takes the evidence with it; a named file survives to be diffed against next week's branch or attached to a pull request. The habit to leave with — every change gets a named run, and `compare_runs` gets the last word.

## Recap

| Concept | One-liner |
|---|---|
| Gold labels | computed by `rules.decide` from the spec; `check_data` recomputes all 40 every build, so data cannot drift. |
| `score_ticket` | three independent axes — decision, policy evidence, amount within $0.01 — and `exact`, their conjunction. |
| `None` semantics | both amounts absent is a match; `None` against a number, or against 0.0, is a miss. |
| Non-dict guard | prose answers and `max_steps` timeouts score zero everywhere instead of crashing the harness. |
| `evaluate` | runs a candidate over a split; four accuracies plus cost; saves `artifacts/runs/<name>.json`. |
| dev / train / test | 12 tickets to iterate on, 20 for fitting, 8 untouched until the end of the course. |
| `compare_runs` | metric deltas between two named runs, plus the ticket ids that flipped. |
| Transcript replay | rerun one failing ticket with `on_step`; the disk cache makes the replay free and faithful. |
| Runs as artifacts | named JSON any process can reload — the substrate of CI for agents. |

## Exercises

1. The dev split comes with no retrieval guarantee: docs/WORLD.md promises the gold policy in the top-2 `search_policy` hits for every *train* ticket only. Evaluate the baseline prompt on the train split as `name="baseline-train"` (20 tickets — roughly ten minutes and a few cents live) and compare the two cards. Is there a `policy_acc` gap between the guaranteed and unguaranteed splits, and which axis moves most?
2. The baseline card leaves no headroom on dev, so attack from the other side: find the smallest system prompt that still ties it. Delete one clause at a time — the tool budget, the one-sentence-commentary rule, the closing "policy, not sympathy" — evaluating each as its own named run (a few cents per variant live, free once cached). Which deletion is the first to move a metric, and on which axis does the damage show first?
3. Add a per-policy breakdown: from `load_run("lazy-prompt")`, group rows by `gold["policy_id"]` and compute exact accuracy per policy. Which policies concentrate the damage, and does that match which cascade branches need data the lazy agent refused to fetch?

**Next up:** chapter 05 stops letting the model drive everything — routing, parallel fan-out, and evaluator loops where plain code, not the model, owns the control flow.